# Training a Logisitic Regression Model on the cleaned Iris Dataset

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import mlflow
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from mlflow.tracking import MlflowClient

from datetime import datetime

/Users/b.ruce6/Desktop/mlops-iris-pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set up MLflow server and experiment

In [2]:
# Set tracking URI to SQLite database (persists across sessions)
# The database will be created automatically if it doesn't exist
# Run mlflow server --backend-store-uri sqlite:///mlflow.db to view.
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
db_path = os.path.join(project_root, "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

# Set the experiment name with date and time
today_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
experiment_name = f"logistic_regression_{today_str}"
mlflow.set_experiment(experiment_name)

print(mlflow.get_tracking_uri())
print(f"Experiment: {experiment_name}")

2026/03/02 00:20:22 INFO mlflow.tracking.fluent: Experiment with name 'logistic_regression_2026-03-02_00-20-22' does not exist. Creating a new experiment.


sqlite:////Users/b.ruce6/Desktop/mlops-iris-pipeline/mlflow.db
Experiment: logistic_regression_2026-03-02_00-20-22


## Load Cleaned Dataset

In [3]:
df = pd.read_csv("../processed_data/iris_cleaned.csv")

# Display the first 5 rows of the dataset
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Train/Validation/Test Split Dataset

We will follow a 60/20/20 split.

In [4]:
# First, split into train (60%) and temp (40%)
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42, shuffle=True)

# Then, split temp into validation (20%) and test (20%)
# Since temp is 40% of total, we split it in half for 20% each
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

# Get X and y for each split
X_train, y_train = train_df.drop("species", axis=1), train_df["species"]
X_val, y_val = val_df.drop("species", axis=1), val_df["species"]
X_test, y_test = test_df.drop("species", axis=1), test_df["species"]

## Train the Logisitic Regression Model w/ Standardization

In [5]:
# Initialize the Logistic Regression model
model = LogisticRegression(random_state=42)

# Create a pipeline
model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', model)
])

# Train the model
model.fit(X_train, y_train)

# Evaluate the model on the validation set
val_score = model.score(X_val, y_val)
print(f"Baseline Validation Accuracy: {val_score:.4f}")

Baseline Validation Accuracy: 1.0000


## Hyperparameter Tuning

In [6]:
# Using validation set to tune hyperparameters
client = MlflowClient()
# Start an MLflow run
with mlflow.start_run():
    C_values = [0.01, 0.1, 1, 10, 100]

    best_val_acc = 0
    best_model = None

    for C in C_values:
        model = LogisticRegression(C=C, random_state=42)

        # Create a pipeline
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('logreg', model)
        ])

        model.fit(X_train, y_train)
        
        y_val_pred = model.predict(X_val)
        val_acc = accuracy_score(y_val, y_val_pred)
        
        print(f"C={C}: Validation Accuracy={val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model

    # Log the model
    model_info = mlflow.sklearn.log_model(sk_model=best_model, artifact_path="model", registered_model_name="logistic_regression_model")
    # Log the loss metric
    mlflow.log_metric("best_val_accuracy", best_val_acc)
    # Get the current run ID
    run_id = mlflow.active_run().info.run_id
    # Find the model version associated with this run
    versions = client.search_model_versions(f"run_id='{run_id}'")
    model_version = versions[0].version
    # Put the metric inside the model
    client.set_model_version_tag(
        name="logistic_regression_model",
        version=model_version,
        key="best_val_accuracy",
        value=str(best_val_acc)
    )

2026/03/02 00:20:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/03/02 00:20:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C=0.01: Validation Accuracy=0.9310
C=0.1: Validation Accuracy=1.0000
C=1: Validation Accuracy=1.0000
C=10: Validation Accuracy=1.0000
C=100: Validation Accuracy=1.0000


Registered model 'logistic_regression_model' already exists. Creating a new version of this model...
Created version '2' of model 'logistic_regression_model'.


## Model Evaluation

In [7]:
# Evaluating the best model on the test set
y_test_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\nBest model test performance:")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_test_pred))


Best model test performance:
Test Accuracy: 0.9333
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.86      1.00      0.92        12
           2       1.00      0.75      0.86         8

    accuracy                           0.93        30
   macro avg       0.95      0.92      0.93        30
weighted avg       0.94      0.93      0.93        30

